In [ ]:
from FIT_python.pipeline_sex.data_import_utils import load_raw_files
from FIT_python.config import RAW_DIR
raw_dfs = load_raw_files(RAW_DIR)
raw_dfs

In [ ]:
# %% [code]
from pathlib import Path
import pandas as pd
import numpy as np
import ast
from tqdm.auto import tqdm
from joblib import Parallel, delayed
from sklearn.model_selection import ParameterSampler

from FIT_python.pipeline_sex.pipeline_wrapper_sex import PipelineWrapper
from FIT_python.config import RESULTS_DATA_DIR

# 1) Prepare (Clean → Split → Summary)
prep = PipelineWrapper(
    model_keys=["log_reg"],  # Dummy
    impute_method=None,
    outlier_method=None,
    scaler_method=None,
    reduce_pre_method=None,
    reduce_post_method=None,
    fs_method=None,
    fs_k=None,
)
prep.prepare()



In [1]:
import shutil
from FIT_python.pipeline_sex.pipeline_wrapper_sex import _cache_dir

# Cache-Verzeichnis komplett entfernen
shutil.rmtree(_cache_dir)
print(f"Cache gelöscht: {_cache_dir}")


Cache gelöscht: C:\Users\Frede\OneDrive\Projects and Disschaptors\FIT_package\results\data\pipeline_cache


In [ ]:
# %% [code] RandomizedGridSearchCV mit Standard-Metriken & Custom-Auswertung (aggregiert)
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from tqdm.auto import tqdm
from tqdm_joblib import tqdm_joblib
import joblib
from functools import reduce
import operator


from FIT_python.pipeline_sex.transform_wrapper                import NumericTransformer
from FIT_python.pipeline_sex.feature_selection_wrapper        import FeatureSelectionTransformer
from FIT_python.pipeline_sex.outlier_wrapper                  import OutlierCleanerTransformer
from FIT_python.pipeline_sex.feature_scaler_wrapper           import FeatureScalerTransformer
from FIT_python.pipeline_sex.dimensionality_reduction_wrapper import DimensionalityReducerTransformer
from FIT_python.pipeline_sex.models                           import MODELS
from FIT_python.pipeline_sex.grouped_metrics                  import individual_accuracies, individual_majority_stats
from FIT_python.config                                        import SPLITS_DIR, RESULTS_DATA_DIR

import warnings
from sklearn.exceptions import ConvergenceWarning

# Warnings nur einmal
warnings.filterwarnings("once", category=ConvergenceWarning, message="Objective did not converge.*")
warnings.filterwarnings("once", category=UserWarning,       message="X does not have valid feature names.*")

# 1) Modell‐Keys
model_keys = [
    "logreg_l2","logreg_l1",
    "rf_small","rf_med","rf_large",
    "et_med","et_sm",
    "knn_3","knn_5","knn_7",
    "svm_linear","svm_rbf",
    "xgb_std","xgb_hist",
    "lgbm_std","lgbm_md10",
    "catb_std","catb_fast","lda",
]

# 2) Param‐Räume (inkl. PCA/UMAP‐ & Outlier‐Hyperparams)
param_distributions = {
    # Feature Selection (inkl. RF)
    "select__method":          [None, "forward", "lasso", "variance", "random_forest"],
    "select__k":               [5, 20, 50],

    # Outlier‐Cleaning
    "outlier__method":         [None, "clip", "zscore"],
    "outlier__lower_quantile": [0.01, 0.05],
    "outlier__upper_quantile": [0.90, 0.95],
    "outlier__z_thresh":       [2.0, 3.0],

    # Scaling
    "scale__method":           [None, "standard", "robust", "minmax"],

    # DimRed Pre
    "reduce_pre__method":        [None, "pca", "umap"],
    "reduce_pre__n_components":  [2, 10],
    "reduce_pre__n_neighbors":   [5, 10, 15, 30, 50],   # ← mehr Nachbarn
    "reduce_pre__min_dist":      [0.1, 0.5],
    "reduce_pre__whiten":        [False, True],

    # DimRed Post
    "reduce_post__method":       [None, "pca", "umap"],
    "reduce_post__n_components": [2, 10],
    "reduce_post__n_neighbors":  [5, 10, 15, 30, 50],   # ← mehr Nachbarn
    "reduce_post__min_dist":     [0.1, 0.5],
    "reduce_post__whiten":       [False, True],

    # Classifier
    "clf": [MODELS[k] for k in model_keys],
}

# 3) Wie groß ist der Suchraum?
lengths = {k: len(v) for k,v in param_distributions.items()}
total_combinations = reduce(operator.mul, lengths.values(), 1)

print("Anzahl der Werte pro Parameter:")
for k, l in lengths.items():
    print(f"  {k:25s}: {l}")
print(f"\n→ Gesamtzahl aller Hyper‐Kombinationen: {total_combinations:,}")


# 3) Ergebnis-Ordner
base_dir = Path(RESULTS_DATA_DIR) / "random_search_standard_metrics"
base_dir.mkdir(parents=True, exist_ok=True)

# Unterordner für Best-Modelle je Metrik
metrics = ["maj_test_pct", "balanced_full_acc", "mean_test_neg_log_loss", "mean_test_accuracy"]
for m in metrics:
    (base_dir / f"best_{m}").mkdir(exist_ok=True)

# 4) Scoring-Dict
scoring = {
    "accuracy":          "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "neg_log_loss":      "neg_log_loss",
}

# 5) Sammel-Listen
raw_records  = []
best_records = []

# 6) Loop über Spezies
for species_dir in tqdm(sorted(Path(SPLITS_DIR).iterdir()), desc="Species"):
    if not species_dir.is_dir(): 
        continue
    species = species_dir.name

    # 6.1) Train/Test laden & Fold entfernen
    df_train = (pd.read_parquet(species_dir / "train.parquet")
                  .query("sex in ['f','m']")
                  .drop(columns=["Fold"], errors="ignore"))
    df_test  = (pd.read_parquet(species_dir / "test.parquet")
                  .query("sex in ['f','m']")
                  .drop(columns=["Fold"], errors="ignore"))

    # 6.2) Features, Ziel & IDs
    X_tr, y_tr, ids_tr = (
        df_train.select_dtypes(include="number"),
        df_train["sex"].map({"f":0,"m":1}).values,
        df_train["individual_id"].values
    )
    X_te, y_te, ids_te = (
        df_test.select_dtypes(include="number"),
        df_test["sex"].map({"f":0,"m":1}).values,
        df_test["individual_id"].values
    )

    # 6.3) Pipeline
    steps = [
        ("transform",  NumericTransformer()),
        ("select",     FeatureSelectionTransformer(method="forward", k=1)),
        ("outlier",    OutlierCleanerTransformer(method="clip")),
        ("scale",      FeatureScalerTransformer(method="standard")),
        ("reduce_pre", DimensionalityReducerTransformer(method=None, n_components=1)),
        ("reduce_post",DimensionalityReducerTransformer(method=None, n_components=1)),
        ("clf",        MODELS["rf_small"]),
    ]
    pipe = Pipeline(steps)

    # 6.4) RandomizedSearchCV
    search = RandomizedSearchCV(
        estimator=pipe,
        param_distributions=param_distributions,
        n_iter=100,      # kleiner Testdurchgang
        scoring=scoring,
        refit=False,
        cv=3,
        n_jobs=-1,
        random_state=42,
        verbose=1,
    )

    # 6.5) Fit mit tqdm
    total_fits = 100 * 3
    with tqdm_joblib(tqdm(desc=f"{species} RS-CV", total=total_fits, leave=False)):
        search.fit(X_tr, y_tr)

    # 6.6) CV-Ergebnisse & Custom-Metriken sammeln
    cv_res = search.cv_results_
    for i, params in enumerate(cv_res["params"]):
        record = {
            "species":                     species,
            "mean_test_accuracy":          cv_res["mean_test_accuracy"][i],
            "mean_test_balanced_accuracy": cv_res["mean_test_balanced_accuracy"][i],
            "mean_test_neg_log_loss":      cv_res["mean_test_neg_log_loss"][i],
            **params
        }
        # kompletter Re-Fit für Custom-Metriken
        mdl = clone(pipe).set_params(**params).fit(X_tr, y_tr)
        y_tr_pred = mdl.predict(X_tr)
        y_te_pred = mdl.predict(X_te)
        y_all_true = np.concatenate([y_tr, y_te])
        y_all_pred = np.concatenate([y_tr_pred, y_te_pred])
        ids_all     = np.concatenate([ids_tr, ids_te])

        fem_tr, mal_tr, bal_tr    = individual_accuracies(y_tr, y_tr_pred, ids_tr)
        fem_te, mal_te, bal_te    = individual_accuracies(y_te, y_te_pred, ids_te)
        ct_tr, wr_tr, pct_tr      = individual_majority_stats(y_tr, y_tr_pred, ids_tr)
        ct_te, wr_te, pct_te      = individual_majority_stats(y_te, y_te_pred, ids_te)
        fem_all, mal_all, bal_all = individual_accuracies(y_all_true, y_all_pred, ids_all)
        ct_all, wr_all, pct_all   = individual_majority_stats(y_all_true, y_all_pred, ids_all)

        record.update({
            "female_train_acc":   fem_tr, "male_train_acc":   mal_tr, "balanced_train_acc": bal_tr,
            "female_test_acc":    fem_te, "male_test_acc":    mal_te, "balanced_test_acc":  bal_te,
            "female_full_acc":    fem_all,"male_full_acc":    mal_all,"balanced_full_acc":  bal_all,
            "maj_train_count":    ct_tr, "maj_train_wrong":  wr_tr, "maj_train_pct":      pct_tr,
            "maj_test_count":     ct_te, "maj_test_wrong":   wr_te, "maj_test_pct":       pct_te,
            "maj_full_count":     ct_all,"maj_full_wrong":   wr_all,"maj_full_pct":       pct_all,
        })
        raw_records.append(record)

    # 6.7) Beste Modelle je Metrik speichern
    df_eval = pd.DataFrame([r for r in raw_records if r["species"]==species])
    for metric in metrics:
        best = df_eval.loc[df_eval[metric].idxmax()].copy()
        best["best_metric"] = metric
        best_records.append(best.to_dict())

        # Modell zur Datei
        best_params = {k: best[k] for k in cv_res["params"][0].keys() if k in best}
        best_pipe = clone(pipe).set_params(**best_params).fit(
            # Zum finalen Training nimm hier wahlweise nur X_tr oder X_tr+X_te
            X_tr, y_tr
        )
        joblib.dump(
            best_pipe,
            base_dir / f"best_{metric}" / f"{species}.joblib"
        )

# 7) Speichern
pd.DataFrame(raw_records).to_csv(base_dir / "all_results.csv", index=False)
pd.DataFrame(best_records).to_csv(base_dir / "best_models.csv", index=False)


Anzahl der Werte pro Parameter:
  select__method           : 5
  select__k                : 3
  outlier__method          : 3
  outlier__lower_quantile  : 2
  outlier__upper_quantile  : 2
  outlier__z_thresh        : 2
  scale__method            : 4
  reduce_pre__method       : 3
  reduce_pre__n_components : 2
  reduce_pre__n_neighbors  : 5
  reduce_pre__min_dist     : 2
  reduce_pre__whiten       : 2
  reduce_post__method      : 3
  reduce_post__n_components: 2
  reduce_post__n_neighbors : 5
  reduce_post__min_dist    : 2
  reduce_post__whiten      : 2
  clf                      : 19

→ Gesamtzahl aller Hyper‐Kombinationen: 393,984,000


Species:   0%|          | 0/10 [00:00<?, ?it/s]

aj_someringii RS-CV:   0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

Fitting 3 folds for each of 100 candidates, totalling 300 fits


...

c:\Users\Frede\anaconda3\envs\FIT_python_conda_env\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\Frede\anaconda3\envs\FIT_python_conda_env\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\Frede\anaconda3\envs\FIT_python_conda_env\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\Frede\anaconda3\envs\FIT_python_conda_env\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\Frede\anaconda3\envs\FIT_python_conda_env\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does n

bengal_tiger RS-CV:   0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

Fitting 3 folds for each of 100 candidates, totalling 300 fits


c:\Users\Frede\anaconda3\envs\FIT_python_conda_env\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 9.188e-02, tolerance: 8.230e-03
  model = cd_fast.enet_coordinate_descent_gram(
c:\Users\Frede\anaconda3\envs\FIT_python_conda_env\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.745e-01, tolerance: 8.230e-03
  model = cd_fast.enet_coordinate_descent_gram(
c:\Users\Frede\anaconda3\envs\FIT_python_conda_env\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale

In [ ]:
# %% [code] Bayesian Hyperparameter-Suche mit scikit-optimize & Custom-Auswertung (aggregiert)
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from tqdm.auto import tqdm
from tqdm_joblib import tqdm_joblib
import joblib
from functools import reduce
import operator
import sklearn.exceptions

from skopt.searchcv import BayesSearchCV
from FIT_python.pipeline_sex.transform_wrapper                import NumericTransformer
from FIT_python.pipeline_sex.feature_selection_wrapper        import FeatureSelectionTransformer
from FIT_python.pipeline_sex.outlier_wrapper                  import OutlierCleanerTransformer
from FIT_python.pipeline_sex.feature_scaler_wrapper           import FeatureScalerTransformer
from FIT_python.pipeline_sex.dimensionality_reduction_wrapper import DimensionalityReducerTransformer
from FIT_python.pipeline_sex.models                           import MODELS
from FIT_python.pipeline_sex.grouped_metrics                  import individual_accuracies, individual_majority_stats
from FIT_python.config                                        import SPLITS_DIR, RESULTS_DATA_DIR


# Warnings nur einmal
warnings.filterwarnings("once", category=ConvergenceWarning, message="Objective did not converge.*")
warnings.filterwarnings("once", category=UserWarning,       message="X does not have valid feature names.*")

# 1) Modell‐Keys
model_keys = [
    "logreg_l2","logreg_l1",
    "rf_small","rf_med","rf_large",
    "et_med","et_sm",
    "knn_3","knn_5","knn_7",
    "svm_linear","svm_rbf",
    "xgb_std","xgb_hist",
    "lgbm_std","lgbm_md10",
    "lda",
]

# 2) Param‐Räume (inkl. PCA/UMAP‐ & Outlier‐Hyperparams)
param_distributions = {
    # Feature Selection (inkl. RF)
    "select__method":          [None, "forward", "lasso", "variance", "random_forest"],
    "select__k":               [5, 20, 50],
    # Outlier‐Cleaning
    "outlier__method":         [None, "clip", "zscore"],
    "outlier__lower_quantile": [0.01, 0.05],
    "outlier__upper_quantile": [0.90, 0.95],
    "outlier__z_thresh":       [2.0, 3.0],
    # Scaling
    "scale__method":           [None, "standard", "robust"],
    # DimRed Pre
    "reduce_pre__method":        [None, "pca", "umap"],
    "reduce_pre__n_components":  [2, 10],
    "reduce_pre__n_neighbors":   [5, 15, 30, 50],
    "reduce_pre__min_dist":      [0.1, 0.5],
    "reduce_pre__whiten":        [False, True],
    # DimRed Post
    "reduce_post__method":       [None, "pca", "umap"],
    "reduce_post__n_components": [2, 10],
    "reduce_post__n_neighbors":  [5, 15, 30, 50],
    "reduce_post__min_dist":     [0.1, 0.5],
    "reduce_post__whiten":       [False, True],
    # Classifier
    "clf": [MODELS[k] for k in model_keys],
}

# 3) Suchraum-Größe bestimmen
lengths = {k: len(v) for k, v in param_distributions.items()}
total_combinations = reduce(operator.mul, lengths.values(), 1)

print("Anzahl der Werte pro Parameter:")
for k, l in lengths.items():
    print(f"  {k:25s}: {l}")
print(f"\n→ Gesamtzahl aller Hyper‐Kombinationen: {total_combinations:,}")

# 4) Ergebnis-Ordner & Unterordner für Best-Modelle je Metrik
base_dir = Path(RESULTS_DATA_DIR) / "bayes_search_standard_metrics"
base_dir.mkdir(parents=True, exist_ok=True)
metrics = ["maj_test_pct", "balanced_full_acc", "mean_test_neg_log_loss", "mean_test_accuracy"]
for m in metrics:
    (base_dir / f"best_{m}").mkdir(exist_ok=True)

# 5) Scoring-Dict
scoring = {
    "accuracy":          "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "neg_log_loss":      "neg_log_loss",
}

# 6) Sammel-Listen
raw_records  = []
best_records = []

# 7) Loop über Spezies
for species_dir in tqdm(sorted(Path(SPLITS_DIR).iterdir()), desc="Species"):
    if not species_dir.is_dir():
        continue
    species = species_dir.name

    # 7.1) Train/Test laden & Fold entfernen
    df_train = (pd.read_parquet(species_dir / "train.parquet")
                  .query("sex in ['f','m']")
                  .drop(columns=["Fold"], errors="ignore"))
    df_test  = (pd.read_parquet(species_dir / "test.parquet")
                  .query("sex in ['f','m']")
                  .drop(columns=["Fold"], errors="ignore"))

    # 7.2) Features, Ziel & IDs
    X_tr, y_tr, ids_tr = (
        df_train.select_dtypes(include="number"),
        df_train["sex"].map({"f":0,"m":1}).values,
        df_train["individual_id"].values
    )
    X_te, y_te, ids_te = (
        df_test.select_dtypes(include="number"),
        df_test["sex"].map({"f":0,"m":1}).values,
        df_test["individual_id"].values
    )

    # 7.3) Pipeline
    steps = [
        ("transform",  NumericTransformer()),
        ("select",     FeatureSelectionTransformer(method="forward", k=1)),
        ("outlier",    OutlierCleanerTransformer(method="clip")),
        ("scale",      FeatureScalerTransformer(method="standard")),
        ("reduce_pre", DimensionalityReducerTransformer(method=None, n_components=1)),
        ("reduce_post",DimensionalityReducerTransformer(method=None, n_components=1)),
        ("clf",        MODELS["rf_small"]),
    ]
    pipe = Pipeline(steps)

    # 7.4) Bayesian Hyperparameter-Suche
    search = BayesSearchCV(
        estimator=pipe,
        search_spaces=param_distributions,
        n_iter=100,                   # Anzahl der Bayesian-Iterationen
        scoring=scoring,
        refit="balanced_accuracy",    # für BayesSearchCV erforderlich bei multi-metric
        cv=3,
        n_jobs=-1,
        random_state=42,
        verbose=1,
    )

    # 7.5) Fit mit tqdm
    total_fits = 100 * 3
    with tqdm_joblib(tqdm(desc=f"{species} Bayes-CV", total=total_fits, leave=False)):
        search.fit(X_tr, y_tr)

    # 7.6) CV-Ergebnisse & Custom-Metriken sammeln
    cv_res = search.cv_results_
    for i, params in enumerate(cv_res["params"]):
        record = {
            "species":                     species,
            "mean_test_accuracy":          cv_res["mean_test_accuracy"][i],
            "mean_test_balanced_accuracy": cv_res["mean_test_balanced_accuracy"][i],
            "mean_test_neg_log_loss":      cv_res["mean_test_neg_log_loss"][i],
            **params
        }
        mdl = clone(pipe).set_params(**params).fit(X_tr, y_tr)
        y_tr_pred = mdl.predict(X_tr)
        y_te_pred = mdl.predict(X_te)
        y_all_true = np.concatenate([y_tr, y_te])
        y_all_pred = np.concatenate([y_tr_pred, y_te_pred])
        ids_all   = np.concatenate([ids_tr, ids_te])

        fem_tr, mal_tr, bal_tr    = individual_accuracies(y_tr,   y_tr_pred,   ids_tr)
        fem_te, mal_te, bal_te    = individual_accuracies(y_te,   y_te_pred,   ids_te)
        ct_tr, wr_tr, pct_tr      = individual_majority_stats(y_tr,   y_tr_pred,   ids_tr)
        ct_te, wr_te, pct_te      = individual_majority_stats(y_te,   y_te_pred,   ids_te)
        fem_all, mal_all, bal_all = individual_accuracies(y_all_true, y_all_pred, ids_all)
        ct_all, wr_all, pct_all   = individual_majority_stats(y_all_true, y_all_pred, ids_all)

        record.update({
            "female_train_acc":   fem_tr, "male_train_acc":   mal_tr, "balanced_train_acc": bal_tr,
            "female_test_acc":    fem_te, "male_test_acc":    mal_te, "balanced_test_acc":  bal_te,
            "female_full_acc":    fem_all,"male_full_acc":    mal_all,"balanced_full_acc":  bal_all,
            "maj_train_count":    ct_tr, "maj_train_wrong":  wr_tr, "maj_train_pct":      pct_tr,
            "maj_test_count":     ct_te, "maj_test_wrong":   wr_te, "maj_test_pct":       pct_te,
            "maj_full_count":     ct_all,"maj_full_wrong":   wr_all,"maj_full_pct":       pct_all,
        })
        raw_records.append(record)

    # 7.7) Beste Modelle je Metrik speichern
    df_eval = pd.DataFrame([r for r in raw_records if r["species"] == species])
    for metric in metrics:
        best = df_eval.loc[df_eval[metric].idxmax()].copy()
        best["best_metric"] = metric
        best_records.append(best.to_dict())

        # Finales Modell trainieren & speichern
        best_params = {k: best[k] for k in cv_res["params"][0].keys() if k in best}
        best_pipe = clone(pipe).set_params(**best_params).fit(X_tr, y_tr)
        joblib.dump(best_pipe, base_dir / f"best_{metric}" / f"{species}.joblib")

# 8) Gesamttabellen speichern
pd.DataFrame(raw_records).to_csv(base_dir / "all_results.csv", index=False)
pd.DataFrame(best_records).to_csv(base_dir / "best_models.csv", index=False)


Anzahl der Werte pro Parameter:
  select__method           : 5
  select__k                : 3
  outlier__method          : 3
  outlier__lower_quantile  : 2
  outlier__upper_quantile  : 2
  outlier__z_thresh        : 2
  scale__method            : 3
  reduce_pre__method       : 3
  reduce_pre__n_components : 2
  reduce_pre__n_neighbors  : 4
  reduce_pre__min_dist     : 2
  reduce_pre__whiten       : 2
  reduce_post__method      : 3
  reduce_post__n_components: 2
  reduce_post__n_neighbors : 4
  reduce_post__min_dist    : 2
  reduce_post__whiten      : 2
  clf                      : 17

→ Gesamtzahl aller Hyper‐Kombinationen: 169,205,760


Species:   0%|          | 0/10 [00:00<?, ?it/s]

c:\Users\Frede\anaconda3\envs\FIT_python_conda_env\Lib\site-packages\skopt\space\space.py:116: UserWarning: Dimension [0.01, 0.05] was inferred to Real(low=0.01, high=0.05, prior='uniform', transform='identity'). In upcoming versions of scikit-optimize, it will be inferred to Categorical(categories=(0.01, 0.05), prior=None). See the documentation of the check_dimension function for the upcoming API.
  warnings.warn(
c:\Users\Frede\anaconda3\envs\FIT_python_conda_env\Lib\site-packages\skopt\space\space.py:116: UserWarning: Dimension [0.9, 0.95] was inferred to Real(low=0.9, high=0.95, prior='uniform', transform='identity'). In upcoming versions of scikit-optimize, it will be inferred to Categorical(categories=(0.9, 0.95), prior=None). See the documentation of the check_dimension function for the upcoming API.
  warnings.warn(
c:\Users\Frede\anaconda3\envs\FIT_python_conda_env\Lib\site-packages\skopt\space\space.py:116: UserWarning: Dimension [2.0, 3.0] was inferred to Real(low=2.0, high

aj_someringii Bayes-CV:   0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

Fitting 3 folds for each of 1 candidates, totalling 3 fits


c:\Users\Frede\anaconda3\envs\FIT_python_conda_env\Lib\site-packages\skopt\space\space.py:116: UserWarning: Dimension [0.01, 0.05] was inferred to Real(low=0.01, high=0.05, prior='uniform', transform='identity'). In upcoming versions of scikit-optimize, it will be inferred to Categorical(categories=(0.01, 0.05), prior=None). See the documentation of the check_dimension function for the upcoming API.
  warnings.warn(
c:\Users\Frede\anaconda3\envs\FIT_python_conda_env\Lib\site-packages\skopt\space\space.py:116: UserWarning: Dimension [0.9, 0.95] was inferred to Real(low=0.9, high=0.95, prior='uniform', transform='identity'). In upcoming versions of scikit-optimize, it will be inferred to Categorical(categories=(0.9, 0.95), prior=None). See the documentation of the check_dimension function for the upcoming API.
  warnings.warn(
c:\Users\Frede\anaconda3\envs\FIT_python_conda_env\Lib\site-packages\skopt\space\space.py:116: UserWarning: Dimension [2.0, 3.0] was inferred to Real(low=2.0, high

Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits


In [ ]:
import pandas as pd
from joblib import load

# Pfade definieren
model_path = "C:/Users/Frede/OneDrive/Projects and Disschaptors/FIT_package/results/data/sex_models_best/eurasian_otter.joblib"
data_path  = "C:/Users/Frede/OneDrive/Projects and Disschaptors/FIT_package/data/raw/Eurasian Otter.csv"
output_csv = "C:/Users/Frede/OneDrive/Projects and Disschaptors/FIT_package/results/data/eurasian_otter_predictions.csv"

# Modell laden
clf = load(model_path)

# Daten laden und Spaltennamen anpassen (Punkte, Bindestriche → Unterstriche; T → t)
df = pd.read_csv(data_path)
df.columns = [
    col.replace('.', '_')
       .replace('-', '_')
       .replace('T', 't')
    for col in df.columns
]

# Direkt Vorhersagen aus der Pipeline holen
df["predicted_sex"]     = clf.predict(df)
probas = clf.predict_proba(df)
df["predicted_proba_f"] = probas[:, 0]
df["predicted_proba_m"] = probas[:, 1]

# Ergebnisse als CSV speichern
df.to_csv(output_csv, index=False)
print(f"Ergebnisse gespeichert in: {output_csv}")
